# Walkthrough: Web Scraping & API Integration

**Pulling data from e-commerce sites with BeautifulSoup + Selenium + XPath + Scrapy & API Integration**

This notebook is the *walkthrough* version of the material in the `steps/` folder. It works well
when run cell by cell while explaining each concept.

**Who it's for:** participants who are new to web scraping & APIs.

**Prerequisites:**
- Understand Python basics (variables, lists, dicts, loops, functions).
- Already ran `uv sync` in the project root (see `README.md`).
- Chrome/Chromium browser installed (for the Selenium section).

**By the end, you will be able to:**
- Understand the flow of a simple data pipeline.
- Scrape static websites with **BeautifulSoup**.
- Scrape dynamic websites with **Selenium** + **XPath**.
- Fetch data from an **API**.
- Clean data with **pandas** and store it in a **database**.


## Outline

0. Setup & imports
1. Understanding the Data Pipeline
2. BeautifulSoup — scrape 1 item
3. BeautifulSoup — 1 page → list of dictionaries
4. BeautifulSoup — multiple pages (pagination)
5. Selenium — dynamic websites
6. XPath
7. API
8. Data cleaning (pandas)
9. Save to database (SQLite)
10. Exercises + pitfalls

### The pipeline we are building

```
Scrape 1 item → Scrape 1 page → List of Dictionaries → Multiple Pages
   → Data Cleaning → Database Connection → Create Table & Insert
```

### Data sources

| Technique              | Website                          |
| ---------------------- | -------------------------------- |
| BeautifulSoup (static) | https://books.toscrape.com       |
| Selenium (dynamic)     | https://quotes.toscrape.com/js   |
| API                    | https://dummyjson.com/products   |


In [1]:
# Setup: import all the libraries used throughout the walkthrough
import re
import sqlite3

import pandas as pd
import requests
from bs4 import BeautifulSoup

print("Libraries ready to use ✅")


Libraries ready to use ✅


## 1. Understanding the Data Pipeline

A **data pipeline** is an **automated** flow for moving data from a **source** to a
**destination**, through a process that is **structured** and **repeatable**.

In this notebook we build a pipeline: fetch data from the web → clean it → store it in a database.

---

## 2. BeautifulSoup — Scrape 1 Item

`BeautifulSoup` is a library for *parsing* HTML, so we can extract data
based on tags/attributes. The two main methods are:

- `find()` → grabs **one** element
- `find_all()` → grabs **multiple** elements

We start with a single item (one book) on `books.toscrape.com`.


In [2]:
# Fetch the page HTML, then parse it with BeautifulSoup
url = "https://books.toscrape.com/"
response = requests.get(url, timeout=10)
response.raise_for_status()
response.encoding = "utf-8"  # so the £ symbol displays correctly

soup = BeautifulSoup(response.text, "html.parser")

# find() -> grab the FIRST item only (<article class="product_pod">)
item = soup.find("article", class_="product_pod")

title = item.find("h3").find("a")["title"]
price = item.find("p", class_="price_color").text
rating = item.find("p", class_="star-rating")["class"][1]
stock = item.find("p", class_="instock availability").text.strip()

print("Title :", title)
print("Price :", price)
print("Rating:", rating)
print("Stock :", stock)


Title : A Light in the Attic
Price : £51.77
Rating: Three
Stock : In stock


## 3. Scrape 1 Page → List of Dictionaries

Now we grab **all** products on a single page with `find_all()`, then store
each product as a **dictionary**, collected into a **list**.


In [3]:
def scrape_page(url: str) -> list[dict]:
    resp = requests.get(url, timeout=10)
    resp.raise_for_status()
    resp.encoding = "utf-8"
    soup = BeautifulSoup(resp.text, "html.parser")

    products = []
    for item in soup.find_all("article", class_="product_pod"):  # ALL books
        products.append(
            {
                "name": item.find("h3").find("a")["title"],
                "price": item.find("p", class_="price_color").text,
                "rating": item.find("p", class_="star-rating")["class"][1],
            }
        )
    return products


one_page_data = scrape_page("https://books.toscrape.com/")
print(f"Got {len(one_page_data)} products")
one_page_data[:3]


Got 20 products


[{'name': 'A Light in the Attic', 'price': '£51.77', 'rating': 'Three'},
 {'name': 'Tipping the Velvet', 'price': '£53.74', 'rating': 'One'},
 {'name': 'Soumission', 'price': '£50.10', 'rating': 'One'}]

## 4. Scrape Multiple Pages (Pagination)

Page N follows the URL pattern `https://books.toscrape.com/catalogue/page-{N}.html`.
We loop over the page numbers and combine the results into one big list.

In [4]:
BASE_URL = "https://books.toscrape.com/catalogue/page-{}.html"


def scrape_multiple_pages(num_pages: int = 3) -> list[dict]:
    all_products = []
    for page in range(1, num_pages + 1):
        url = BASE_URL.format(page)
        print(f"-> scraping page {page}")
        all_products.extend(scrape_page(url))
    return all_products


raw_data = scrape_multiple_pages(num_pages=3)
print(f"Total {len(raw_data)} products from multiple pages")
raw_data[:3]

-> scraping page 1


-> scraping page 2


-> scraping page 3


Total 60 products from multiple pages


[{'name': 'A Light in the Attic', 'price': '£51.77', 'rating': 'Three'},
 {'name': 'Tipping the Velvet', 'price': '£53.74', 'rating': 'One'},
 {'name': 'Soumission', 'price': '£50.10', 'rating': 'One'}]

## 5. Selenium — Dynamic Websites

Some websites build their content with **JavaScript**. For example
`https://quotes.toscrape.com/js/` — if you fetch it with `requests`, the content is empty.

`Selenium` controls a **real browser**, so JavaScript runs as well.

> The cell below will **open a Chrome window** briefly (default `headless=False` so it is
> visible during the demo). The driver is downloaded automatically by **Selenium Manager**, so
> there is no need to install ChromeDriver manually. Set `headless=True` if you don't want a window to open.

In [5]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By


def build_driver(headless: bool = False) -> webdriver.Chrome:
    options = Options()
    if headless:
        options.add_argument("--headless=new")
    options.add_argument("--window-size=1920,1080")
    return webdriver.Chrome(options=options)


driver = build_driver()
try:
    driver.get("https://quotes.toscrape.com/js/")
    quotes = driver.find_elements(By.CLASS_NAME, "quote")
    print(f"Got {len(quotes)} quotes from the dynamic website")
    for q in quotes[:3]:
        text = q.find_element(By.CLASS_NAME, "text").text
        author = q.find_element(By.CLASS_NAME, "author").text
        print(f"- {text}  -- {author}")
finally:
    driver.quit()  # always close the browser

Got 10 quotes from the dynamic website
- “The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”  -- Albert Einstein
- “It is our choices, Harry, that show what we truly are, far more than our abilities.”  -- J.K. Rowling
- “There are only two ways to live your life. One is as though nothing is a miracle. The other is as though everything is a miracle.”  -- Albert Einstein


## 6. XPath

**XPath** is a query language for selecting elements inside the DOM (HTML/XML).
You can try it at [xpath-playground](https://scrapinghub.github.io/xpath-playground/).

A few examples:

| XPath                          | Meaning                                    |
| ------------------------------ | ------------------------------------------ |
| `//h1`                         | all `<h1>` tags                            |
| `//p[1]`                       | the first `<p>` tag                        |
| `//*[@id="first-name"]`        | the element with `id="first-name"`         |
| `//p[@class="plot"]`           | `<p>` with class exactly `plot`            |
| `//p[contains(@class,"plot")]` | `<p>` whose class **contains** `plot`      |

In Selenium, we use `By.XPATH` to find elements.

In [6]:
driver = build_driver()
try:
    driver.get("https://quotes.toscrape.com/js/")

    # XPath: all <div class="quote">
    quotes = driver.find_elements(By.XPATH, '//div[@class="quote"]')
    print(f"Got {len(quotes)} quotes via XPath")

    for q in quotes[:3]:
        # Relative XPath (starts with ".") -> searched WITHIN this quote element
        text = q.find_element(By.XPATH, './/span[@class="text"]').text
        author = q.find_element(By.XPATH, './/small[@class="author"]').text
        print(f"- {text}  -- {author}")

    print("\nPage title (//h1):", driver.find_element(By.XPATH, "//h1").text)
finally:
    driver.quit()

Got 0 quotes via XPath

Page title (//h1): Quotes to Scrape


## 7. API

An **API** is the official way to fetch structured data directly from the provider's system.
The data is usually in **JSON** form, stable, and faster than scraping.

We use a public e-commerce API: `https://dummyjson.com/products`.

In [7]:
resp = requests.get(
    "https://dummyjson.com/products",
    params={"limit": 5, "select": "title,price,category,brand"},
    timeout=10,
)
resp.raise_for_status()

api_data = resp.json()  # convert JSON into a Python dict
api_products = api_data["products"]
print(f"Got {len(api_products)} products from the API")
api_products

Got 5 products from the API


[{'id': 1,
  'title': 'Essence Mascara Lash Princess',
  'price': 9.99,
  'category': 'beauty',
  'brand': 'Essence'},
 {'id': 2,
  'title': 'Eyeshadow Palette with Mirror',
  'price': 19.99,
  'category': 'beauty',
  'brand': 'Glamour Beauty'},
 {'id': 3,
  'title': 'Powder Canister',
  'price': 14.99,
  'category': 'beauty',
  'brand': 'Velvet Touch'},
 {'id': 4,
  'title': 'Red Lipstick',
  'price': 12.99,
  'category': 'beauty',
  'brand': 'Chic Cosmetics'},
 {'id': 5,
  'title': 'Red Nail Polish',
  'price': 8.99,
  'category': 'beauty',
  'brand': 'Nail Couture'}]

## 8. Data Cleaning (pandas)

The scraped data is still "dirty": price is text `"£51.77"`, rating is a word `"Three"`.
We clean it with **pandas** before saving:

1. Load it into a `DataFrame`
2. Convert price `"£51.77"` → `51.77` (float)
3. Convert rating `"Three"` → `3` (int)
4. Drop empty rows & duplicates

In [8]:
RATING_MAP = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}

df = pd.DataFrame(raw_data)

# price: "£51.77" -> 51.77 (strip every character except digits & the dot)
df["price"] = df["price"].apply(lambda x: re.sub(r"[^0-9.]", "", x)).astype(float)

# rating: text -> number
df["rating"] = df["rating"].map(RATING_MAP)

# drop empty rows & duplicates
df = df.dropna().drop_duplicates().reset_index(drop=True)

print("Data type of each column:")
print(df.dtypes)
df.head()

Data type of each column:
name          str
price     float64
rating      int64
dtype: object


,name,price,rating
0,A Light in the Attic,51.77,3
1,Tipping the Velvet,53.74,1
2,Soumission,50.10,1
3,Sharp Objects,47.82,4
4,Sapiens: A Brief History of Humankind,54.23,5


## 9. Save to Database (SQLite)

The final step: save the clean data to a database. We use **SQLite** (built into Python,
no server to install). The flow is: **create connection → create table → insert data**.

> A **PostgreSQL** version (`psycopg2`) is in `steps/09_simpan_database.py` if you want
> to demonstrate a database server.

In [9]:
DB_PATH = "walkthrough.db"

# 1) create the connection
conn = sqlite3.connect(DB_PATH)

# 2) create the table
conn.execute(
    """
    CREATE TABLE IF NOT EXISTS products (
        id     INTEGER PRIMARY KEY AUTOINCREMENT,
        name   TEXT NOT NULL,
        price  REAL,
        rating INTEGER
    )
    """
)
conn.execute("DELETE FROM products")  # clear it first to avoid duplicates on re-run

# 3) insert the data (using the cleaned DataFrame)
df.to_sql("products", conn, if_exists="append", index=False)
conn.commit()

# check the result: read it back from the database
result = pd.read_sql("SELECT * FROM products LIMIT 5", conn)
conn.close()
print(f"Saved to {DB_PATH}")
result

Saved to walkthrough.db


,id,name,price,rating
0,1,A Light in the Attic,51.77,3
1,2,Tipping the Velvet,53.74,1
2,3,Soumission,50.10,1
3,4,Sharp Objects,47.82,4
4,5,Sapiens: A Brief History of Humankind,54.23,5


## 10. Exercises

1. Modify `scrape_multiple_pages` to fetch **5 pages**, then compute the **average price**
   of the books using pandas.
2. (Bonus) Add a **link** column to the scraped results, then save it to the database table.

Try it yourself before looking at the example answer in the next cell.

In [10]:
# Example answer for exercise 1
data_5 = scrape_multiple_pages(num_pages=5)

df_5 = pd.DataFrame(data_5)
df_5["price"] = df_5["price"].apply(lambda x: re.sub(r"[^0-9.]", "", x)).astype(float)

print(f"Number of books : {len(df_5)}")
print(f"Average price: £{df_5['price'].mean():.2f}")

-> scraping page 1


-> scraping page 2


-> scraping page 3


-> scraping page 4


-> scraping page 5


Number of books : 100
Average price: £34.56


## Pitfalls & Extensions

**Common mistakes:**
- Forgetting to check `response.raise_for_status()` → silently processing an error page.
- `find()` returns `None` when the element is missing → it will error on `.text`. Always make sure the structure is correct.
- Forgetting `driver.quit()` → many browser processes stuck in memory.
- Scraping too fast / aggressively → respect the site (add delays, read `robots.txt`).

**Extensions:**
- Run the full pipeline from the terminal: `uv run python pipeline.py --pages 5`.
- Add `time.sleep()` between requests, or use `WebDriverWait` to wait for elements to appear.
- Save to **PostgreSQL** (see `steps/09_simpan_database.py`).